<a href="https://colab.research.google.com/github/manoj96-alt/LoRA/blob/main/day12_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from openai import OpenAI

# Initialize LLM
client = OpenAI()

st.title("📄 AI Document Chatbot")

mode = st.selectbox("Choose mode", ["Explain", "Summarize", "Bullet Points"])


# Load and process document (runs once)
@st.cache_resource
def load_rag_system():

    reader = PdfReader("data/documents/sample.pdf")

    text = ""
    for page in reader.pages:
        text += page.extract_text()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_text(text)

    embeddings = OpenAIEmbeddings()

    vector_db = FAISS.from_texts(chunks, embeddings)

    return vector_db

vector_db = load_rag_system()

# User input
query = st.text_input("Ask a question about the document:")

if query:

    results = vector_db.similarity_search(query, k=5)

    context = "\n\n".join([doc.page_content for doc in results])
    context = context[:2000]

    prompt = f"""
You are an expert assistant.

Follow these rules:
1. Answer ONLY using the context
2. If unsure, say "I don't know"
3. Be clear and concise

Context:
{context}

Question:
{query}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    st.subheader("Answer:")
    st.write(response.output_text)

    st.subheader("Retrieved Context")
    st.write(context[:500])

